### Pinecone Vector DB

In [25]:
import os
api_key=os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")

In [26]:
!pip install -qU langchain langchain-pinecone langchain-openai

In [27]:
from pinecone import Pinecone
pc=Pinecone(api_key=api_key)


In [28]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024, api_key=OPENAI_API_KEY)

In [29]:
from pinecone import ServerlessSpec

index_name = "rag"  #change if desired

if not pc.has_index(index_name):
  pc.create_index(
      name=index_name,
      dimension=1024,
      metric="cosine",
      spec=ServerlessSpec(cloud="aws", region="us-east-1")
  )

index = pc.Index(index_name)

print("Connected to Pinecone database")

Connected to Pinecone database


In [30]:
index

In [31]:
from langchain_pinecone import PineconeVectorStore

vectorstore = PineconeVectorStore(index=index, embedding=embeddings)
vectorstore

In [32]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"}
)
document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"}
)
document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"}
)

document_4 = Document(
    page_content="Robbers broke into the city back and stole $1 million in cash and jewelry.",
    metadata={"source": "news"}
)

document_5 = Document(
    page_content="WoW! That was an amazing movie. I can't wait to see it again!",
    metadata={"source": "tweet"}
)

document_6 = Document(
    page_content="Is the new IPhone worth the price? Read this review to find out.",
    metadata={"source": "website"}
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now. Who do you think is the best?",
    metadata={"source": "website"}
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications with LLMs.",
    metadata={"source": "tweet"}
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"}
)

document_10 = Document(
    page_content="Just finished a great workout at the gym! Feeling strong and healthy.",
    metadata={"source": "tweet"}
)

documents = [document_1, document_2, document_3, document_4, document_5, document_6, document_7, document_8, document_9, document_10]
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city back and stole $1 million in cash and jewelry.'),
 Document(metadata={'source': 'tweet'}, page_content="WoW! That was an amazing movie. I can't wait to see it again!"),
 Document(metadata={'source': 'website'}, page_content='Is the new IPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now. Who do you think is the best?'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework 

In [33]:
vectorstore.add_documents(documents=documents)

['c8ac8dd1-70a9-42af-afe7-fadc07d0ee76',
 '0c15f4d3-0db2-4b76-9f6f-c5d959bf606e',
 'c5ae59fa-56f0-468c-be4c-5380f6d1f1e8',
 '0a95202d-2308-42d0-86b8-f934e6b77eb9',
 '4a0c1fe6-a2e6-4f5a-b5ae-8a6e130f9fa9',
 'c1ed2acf-3120-4ecb-a565-8b8761ecbb4b',
 '8e3e44e2-8e29-4761-9f1f-d6f4e11b793d',
 '47c66de9-db68-4486-a28c-6d2d48d5e9a1',
 'a00fe95c-508c-4a6a-bb2f-e2aea537c893',
 '17953ae0-e956-488e-ac4c-266bc22f1558']

In [34]:
print("All the documents are inserted")

All the documents are inserted


In [36]:
# Query directly
results = vectorstore.similarity_search("LangChain provides abstractions to make working with LLMs easy",
                                        k=3,
                                        filter={"source":"tweet"}
                                        )
for res in results:
  print(f'* "{res.page_content}", metadata={res.metadata}')

* "LangGraph is the best framework for building stateful, agentic applications with LLMs.", metadata={'source': 'tweet'}
* "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}
* "WoW! That was an amazing movie. I can't wait to see it again!", metadata={'source': 'tweet'}


In [39]:
results = vectorstore.similarity_search_with_score("Will it be hot tomorrow?",
                                        k=1,
                                        filter={"source":"news"}
                                        )
for res, score in results:
  print(f'* [SIM={score:.3f}] "{res.page_content}", metadata={res.metadata}')

* [SIM=0.573] "The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.", metadata={'source': 'news'}


In [40]:
# Retriever

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.5}
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='0a95202d-2308-42d0-86b8-f934e6b77eb9', metadata={'source': 'news'}, page_content='Robbers broke into the city back and stole $1 million in cash and jewelry.')]